In [ ]:
import pandas as pd
import time
import numpy as np
from datasets import load_dataset
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.preprocessing import LabelEncoder

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, accuracy_score
import numpy as np
import time

# ── 1. Load ────────────────────────────────────────────────────────────────
ds = load_dataset("bvsam/cic-ids-2017", "machine_learning")
df = ds["train"].to_pandas()

# ── 2. Quick look at label distribution ───────────────────────────────────
print("Label counts:\n", df["Label"].value_counts())

# ── 3. Sample 10% for speed ────────────────────────────────────────────────
df_sample = df.sample(frac=0.1, random_state=42).copy()

# ── 4. Drop non-feature columns ────────────────────────────────────────────
drop_cols = ["Flow ID", "Source IP", "Destination IP", "Timestamp"]
df_sample.drop(columns=drop_cols, errors="ignore", inplace=True)

# ── 5. Binary label (keep original Label too for Stage 2) ─────────────────
df_sample["binary_label"] = df_sample["Label"].apply(
    lambda x: 0 if x == "BENIGN" else 1   # 0 = Normal, 1 = Malicious
)

# ── 6. Features / target ──────────────────────────────────────────────────
X = df_sample.drop(columns=["Label", "binary_label"])
y_binary = df_sample["binary_label"]

# Handle NaN and inf (common in CIC-IDS2017!)
X.replace([np.inf, -np.inf], np.nan, inplace=True)
X.fillna(0, inplace=True)

# ── 7. Train/test split ───────────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y_binary, test_size=0.2, random_state=42, stratify=y_binary
)

# ── 8. Train Decision Tree ────────────────────────────────────────────────
dt = DecisionTreeClassifier(max_depth=7, random_state=42)
dt.fit(X_train, y_train)

# ── 9. Predict + measure response time ───────────────────────────────────
start = time.time()
y_pred = dt.predict(X_test)
dt_response_time = (time.time() - start) / len(X_test)  # per sample

# ── 10. Metrics ───────────────────────────────────────────────────────────
cm = confusion_matrix(y_test, y_pred)
TN, FP, FN, TP = cm.ravel()

detection_rate = TP / (TP + FN)          # Recall
false_positive_rate = FP / (FP + TN)

print(f"\n── Stage 1: Decision Tree ──────────────────")
print(f"Accuracy:          {accuracy_score(y_test, y_pred):.4f}")
print(f"Detection Rate:    {detection_rate:.4f}")
print(f"False Pos. Rate:   {false_positive_rate:.4f}")
print(f"Response Time:     {dt_response_time*1000:.4f} ms/sample")
print(f"Confusion Matrix:\n{cm}")

# ── 11. Save malicious samples for Stage 2 ───────────────────────────────
# We need the ORIGINAL multi-class labels for FT-Transformer input
malicious_mask_test = y_pred == 1   # what DT flagged as malicious
X_malicious = X_test[malicious_mask_test].copy()
y_malicious_true = df_sample.loc[X_malicious.index, "Label"]  # original attack type

print(f"\nMalicious samples passed to Stage 2: {len(X_malicious)}")
print("Attack types:\n", y_malicious_true.value_counts())

# ── 1. Prepare data from Stage 1 ─────────────────────────────────────────
X_stage2 = X_malicious.copy()
y_stage2 = y_malicious_true.copy()

# Drop any BENIGN that slipped through
# mask = y_stage2 != "BENIGN"
# X_stage2 = X_stage2[mask]
# y_stage2 = y_stage2[mask]

# --- FIX: Filter out classes with only one sample before Label Encoding and splitting ---
class_counts = y_stage2.value_counts()
single_sample_classes = class_counts[class_counts == 1].index

if not single_sample_classes.empty:
    print(f"Warning: Dropping classes with single samples: {list(single_sample_classes)}")
    mask_multi_sample = ~y_stage2.isin(single_sample_classes)
    X_stage2 = X_stage2[mask_multi_sample]
    y_stage2 = y_stage2[mask_multi_sample]

le = LabelEncoder()
y_encoded = le.fit_transform(y_stage2)
num_classes = len(le.classes_)
print(f"Attack classes ({num_classes}): {list(le.classes_)}")

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_stage2)

X_tr, X_te, y_tr, y_te = train_test_split(
    X_scaled, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

def to_tensor(X, y):
    return torch.tensor(X, dtype=torch.float32).to(device), \
           torch.tensor(y, dtype=torch.long).to(device)

X_tr_t, y_tr_t = to_tensor(X_tr, y_tr)
X_te_t, y_te_t = to_tensor(X_te, y_te)

train_loader = DataLoader(TensorDataset(X_tr_t, y_tr_t), batch_size=256, shuffle=True)
test_loader  = DataLoader(TensorDataset(X_te_t, y_te_t), batch_size=256, shuffle=False)

# ── 2. FT-Transformer built from scratch ─────────────────────────────────
class FeatureTokenizer(nn.Module):
    """Embeds each continuous feature into a d_token-dim vector (like FT-Transformer paper)"""
    def __init__(self, n_features, d_token):
        super().__init__()
        # Each feature gets its own weight + shared bias
        self.weight = nn.Parameter(torch.empty(n_features, d_token))
        self.bias   = nn.Parameter(torch.empty(n_features, d_token))
        nn.init.kaiming_uniform_(self.weight)
        nn.init.zeros_(self.bias)
        # Learnable [CLS] token (BERT-style)
        self.cls_token = nn.Parameter(torch.zeros(1, 1, d_token))

    def forward(self, x):
        # x: (batch, n_features) → (batch, n_features, d_token)
        tokens = x.unsqueeze(-1) * self.weight.unsqueeze(0) + self.bias.unsqueeze(0)
        # Prepend [CLS] token
        cls = self.cls_token.expand(x.size(0), -1, -1)
        return torch.cat([cls, tokens], dim=1)   # (batch, n_features+1, d_token)


class FTTransformer(nn.Module):
    def __init__(self, n_features, d_token=64, n_heads=8, n_layers=3,
                 ffn_factor=4, dropout=0.1, n_classes=2):
        super().__init__()
        self.tokenizer = FeatureTokenizer(n_features, d_token)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_token,
            nhead=n_heads,
            dim_feedforward=d_token * ffn_factor,
            dropout=dropout,
            batch_first=True,      # (batch, seq, features)
            norm_first=True        # Pre-LN like the original paper
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.norm = nn.LayerNorm(d_token)
        self.head = nn.Linear(d_token, n_classes)

    def forward(self, x):
        tokens = self.tokenizer(x)           # (batch, n_features+1, d_token)
        out    = self.transformer(tokens)    # (batch, n_features+1, d_token)
        cls    = self.norm(out[:, 0, :])     # take [CLS] token only — BERT encoder style
        return self.head(cls)                # (batch, n_classes)


n_features = X_tr_t.shape[1]
model = FTTransformer(
    n_features=n_features,
    d_token=64,      # embedding dim per feature
    n_heads=8,
    n_layers=3,
    dropout=0.1,
    n_classes=num_classes
).to(device)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model parameters: {total_params:,}")

# ── 3. Training ───────────────────────────────────────────────────────────
optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-5)
criterion = nn.CrossEntropyLoss()
EPOCHS = 15

print("\n── Training FT-Transformer ──────────────────")
for epoch in range(EPOCHS):
    model.train()
    epoch_loss, correct, total = 0, 0, 0
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        logits = model(X_batch)
        loss   = criterion(logits, y_batch)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
        correct    += (logits.argmax(1) == y_batch).sum().item()
        total      += y_batch.size(0)

    print(f"  Epoch {epoch+1:2d}/{EPOCHS} | "
          f"Loss: {epoch_loss/len(train_loader):.4f} | "
          f"Train Acc: {correct/total:.4f}")

# ── 4. Evaluation ─────────────────────────────────────────────────────────
model.eval()
all_preds, all_labels = [], []

start = time.time()
with torch.no_grad():
    for X_batch, y_batch in test_loader:
        preds = model(X_batch).argmax(1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(y_batch.cpu().numpy())
ft_response_time = (time.time() - start) / len(X_te)

all_preds  = np.array(all_preds)
all_labels = np.array(all_labels)

cm  = confusion_matrix(all_labels, all_preds)
acc = accuracy_score(all_labels, all_preds)

# ── 5. Per-class & macro metrics ─────────────────────────────────────────
print(f"\n── Per-class Results ────────────────────────")
detection_rates, false_positive_rates = [], []

for i in range(num_classes):
    TP = cm[i, i]
    FN = cm[i, :].sum() - TP
    FP = cm[:, i].sum() - TP
    TN = cm.sum() - TP - FN - FP
    dr  = TP / (TP + FN) if (TP + FN) > 0 else 0
    fpr = FP / (FP + TN) if (TP + TN) > 0 else 0
    detection_rates.append(dr)
    false_positive_rates.append(fpr)
    print(f"  {le.classes_[i]:30s}  DR: {dr:.4f}  FPR: {fpr:.4f}")

# ── 6. Full pipeline summary ──────────────────────────────────────────────
print(f"\n══ Full Pipeline Summary ════════════════════")
print(f"{'Metric':<28} {'Stage 1: DT':>14} {'Stage 2: FT-T':>14}")
print(f"{'─'*58}")
print(f"{'Detection Rate':<28} {detection_rate:>14.4f} {np.mean(detection_rates):>14.4f}")
print(f"{'False Positive Rate':<28} {false_positive_rate:>14.4f} {np.mean(false_positive_rates):>14.4f}")
print(f"{'Response Time (ms/sample)':<28} {dt_response_time*1000:>14.4f} {ft_response_time*1000:>14.4f}")
print(f"{'─'*58}")
print(f"{'Total Pipeline Response Time':<28} {(dt_response_time+ft_response_time)*1000:>14.4f} ms/sample")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

machine_learning/Friday-WorkingHours-Aft(…):   0%|          | 0.00/22.2M [00:00<?, ?B/s]

machine_learning/Friday-WorkingHours-Aft(…):   0%|          | 0.00/15.7M [00:00<?, ?B/s]

machine_learning/Friday-WorkingHours-Mor(…):   0%|          | 0.00/19.4M [00:00<?, ?B/s]

...

machine_learning/Wednesday-workingHours.(…):   0%|          | 0.00/67.4M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/2830743 [00:00<?, ? examples/s]

Label counts:
 Label
BENIGN                        2273097
DoS Hulk                       231073
PortScan                       158930
DDoS                           128027
DoS GoldenEye                   10293
FTP-Patator                      7938
SSH-Patator                      5897
DoS slowloris                    5796
DoS Slowhttptest                 5499
Bot                              1966
Web Attack � Brute Force         1507
Web Attack � XSS                  652
Infiltration                       36
Web Attack � Sql Injection         21
Heartbleed                         11
Name: count, dtype: int64

── Stage 1: Decision Tree ──────────────────
Accuracy:          0.9928
Detection Rate:    0.9779
False Pos. Rate:   0.0036
Response Time:     0.0005 ms/sample
Confusion Matrix:
[[45296   163]
 [  246 10910]]

Malicious samples passed to Stage 2: 11073
Attack types:
 Label
DoS Hulk            4654
PortScan            3148
DDoS                2504
DoS GoldenEye        177
BENIGN   

/tmp/ipykernel_247/3652099039.py:163: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)


Model parameters: 160,713

── Training FT-Transformer ──────────────────
  Epoch  1/15 | Loss: 1.1938 | Train Acc: 0.6947
  Epoch  2/15 | Loss: 0.4899 | Train Acc: 0.9172
  Epoch  3/15 | Loss: 0.3412 | Train Acc: 0.9291
  Epoch  4/15 | Loss: 0.2598 | Train Acc: 0.9408
  Epoch  5/15 | Loss: 0.2073 | Train Acc: 0.9595
  Epoch  6/15 | Loss: 0.1687 | Train Acc: 0.9717
  Epoch  7/15 | Loss: 0.1464 | Train Acc: 0.9778
  Epoch  8/15 | Loss: 0.1435 | Train Acc: 0.9772
  Epoch  9/15 | Loss: 0.1180 | Train Acc: 0.9828
  Epoch 10/15 | Loss: 0.1022 | Train Acc: 0.9859
  Epoch 11/15 | Loss: 0.0951 | Train Acc: 0.9858
  Epoch 12/15 | Loss: 0.0892 | Train Acc: 0.9860
  Epoch 13/15 | Loss: 0.0829 | Train Acc: 0.9855
  Epoch 14/15 | Loss: 0.0794 | Train Acc: 0.9861
  Epoch 15/15 | Loss: 0.0740 | Train Acc: 0.9875

── Per-class Results ────────────────────────
  BENIGN                          DR: 0.3939  FPR: 0.0000
  DDoS                            DR: 1.0000  FPR: 0.0000
  DoS GoldenEye              